# ERM 424 · Module 3 · Clean It Twice

A second extract has arrived from the claims organization: a new claims file and a policyholder reference table. Neither is trustworthy alone. You will produce a defensible cleaned dataset twice, by two routes, and the comparison between the routes is part of what you are learning.

- **Part A, manual** (about 3 hours, 42 points): join the tables, investigate the missingness before you treat anything, clean in code, run the imputation-distortion experiment, record every decision in a log, then re-profile and write three sentences.
- **Part B, AI-assisted** (about 1 hour, 18 points): an AI assistant proposes a cleaning script for the same raw tables, and you run it and audit it. The audit is graded, not the AI's output.

**Before anything else: File, then Save a copy in Drive.** Work in your copy.

Course objectives served: CO1, CO2, CO3, CO5, CO6.

Version 1.2, 2026-11-05. Data dictionary: [`m3_DICTIONARY.md`](https://github.com/nybabot/ERM424/blob/main/data/claims/m3_DICTIONARY.md) in the course repository.

In [ ]:
# Setup: run this cell first.
import sys, warnings
import numpy as np
import pandas as pd
EXPECTED = {"pandas": "2.2", "numpy": "2.1"}
for name, want in EXPECTED.items():
    have = __import__(name).__version__
    if not have.startswith(want):
        print(f"Note: {name} is {have}; this notebook was built on {want}.x. Most cells will still run.")
%pip install -q fg-data-profiling==4.20.0
# Add-on pins verified on the Colab runtime on 2026-10-15 (Python 3.13.16, pandas 2.2.3, numpy 2.1.3); see OPEN_QUESTIONS Q04 and Q10.
import os, time
import matplotlib.pyplot as plt
# Chart colors: the Okabe-Ito palette, readable with most forms of color-vision difference. Meaning never rides on color alone.
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=OKABE_ITO)
plt.rcParams["figure.dpi"] = 100

# Where the course files live. By default (Colab) the notebook reads them from the course repository on GitHub.
# If the course folders sit next to this notebook (a local copy of the repository), LOCAL becomes True and it reads them from disk.
REPO = "nybabot/ERM424"   # course repository on GitHub, as owner/name
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/claims/"
GENERATOR_URL = f"https://raw.githubusercontent.com/{REPO}/main/generators/claims_world.py"
LOCAL = False
DATA_DIR = "../../data/claims"   # notebooks/solutions, notebooks/student and docs/sync sit two folders below the course root
GEN_DIR = "../../generators"
for _cand in ["../../", "../", "./", "/content/"]:   # also works from qa/, from the root, or on a Colab VM with the folders under /content
    if os.path.isdir(_cand + "data/claims"):
        DATA_DIR, GEN_DIR = _cand + "data/claims", _cand + "generators"
        LOCAL = True
        break

def load(name, **kwargs):
    """Read one course CSV from the local data folder (LOCAL) or from the course repository (published)."""
    source = os.path.join(DATA_DIR, name) if LOCAL else BASE_URL + name
    return pd.read_csv(source, **kwargs)

print("pandas", pd.__version__, "| numpy", np.__version__, "| reading data from", os.path.abspath(DATA_DIR) if LOCAL else BASE_URL)


In [ ]:
# Load the claims extract and look at the first rows.
claims = load("m3_claims_extract.csv")
print(claims.shape[0], "rows,", claims.shape[1], "columns")
claims.head()

In [ ]:
# Load the policyholder reference table and look at the first rows.
policyholders = load("m3_policyholders.csv")
print(policyholders.shape[0], "rows,", policyholders.shape[1], "columns")
policyholders.head()

The two tables are `claims` and `policyholders`. Read the data dictionary before you do anything else; every rule it states is worth checking. Treat both files as they arrive: nothing in this notebook edits them. All cleaning happens in code, on copies, and nothing is changed by hand.

In [ ]:
# Helper: report whether a table you build in an earlier step exists yet, so later cells never fail on a blank notebook.
def ready(value, step):
    """Return True if value has been built; otherwise print what to build first."""
    if value is None:
        print(f"Not built yet: finish {step} first. This cell runs once you have.")
        return False
    return True

## Part A: Manual

About 3 hours. The order matters: join first, investigate before you treat, then clean, experiment, log, and re-profile. Everything is code; nothing is hand-edited.

Cells marked `# YOUR CODE HERE` are yours to write. Every other cell runs as it is. A cell that needs a table you have not built yet prints a one-line message instead of failing, so you can run the whole notebook at any point.

Keep the decision log (Step A5) open as you work. It is the assignment's central artifact.

### Step A1: Join the tables

Combine `claims` with `policyholders` on `policy_id`, keeping every claim (a left join). Use `indicator=True` so every row records where it came from. Build the joined table as `joined_raw`.

Before you join, check the key on both sides: is `policy_id` unique where it should be, and does every claim have a match? Then compare row counts before and after. The join has consequences; notice them and record them.

In [ ]:
# Template: the joined table you build in this step. Later cells wait for it.
joined_raw = None

In [ ]:
# Key integrity: duplicated keys in the reference table, and claims whose key is absent from it.
# YOUR CODE HERE

In [ ]:
# The join: left join the claims to the reference table on policy_id, with a merge indicator, and compare row counts.
# YOUR CODE HERE

In [ ]:
# Given: rows and distinct claim numbers before and after the join.
if ready(joined_raw, "the join in Step A1"):
    before_after = pd.DataFrame({"rows": [len(claims), len(joined_raw)],
                                 "distinct claim_id": [claims["claim_id"].nunique(), joined_raw["claim_id"].nunique()]},
                                index=["claims (before)", "joined (after)"])
    display(before_after)

Compare the row count before and after the join. Does it match what you expected? If not, what explains it; if so, what did you check to be sure? What would you check next?

> Your answer:

### Step A2: Investigate the missingness before you treat anything

Before you touch a single blank, find out where values are missing and whether the pattern is explainable by other columns. Some of what looks random in one table stops looking random once the tables are combined.

The reading names three mechanisms: **missing completely at random** (MCAR), where the gaps have no relation to anything in the data; **missing at random** (MAR), where the gaps are explained by other observed columns; and **missing not at random** (MNAR), where the gaps depend on the missing value itself. Often the honest answer is a fourth: the data alone cannot tell.

Compare blank rates across groups and always look at the size of each group next to its rate. A difference between two groups of 20 rows means little; the same difference between two groups of 300 rows means more.

Here is the pattern on an invented table. `blank_rate` shows the blank count, the group size and the blank rate together. `crosstab` shows the same counts as a grid. Use `blank_rate` in the cells below.

In [ ]:
# Worked pattern on an invented table: a blank-rate helper that always shows group sizes, and a crosstab.
def blank_rate(frame, column, by):
    """Blank count, group size and blank rate of one column, grouped by one column or series."""
    groups = frame.groupby(by, dropna=False, observed=True)[column]
    out = pd.DataFrame({"blank": groups.apply(lambda s: s.isna().sum()), "rows": groups.size()})
    out["blank_rate"] = (out["blank"] / out["rows"]).round(3)
    return out

demo = pd.DataFrame({
    "plan": ["Basic"] * 6 + ["Plus"] * 6,
    "branch": ["North", "South"] * 6,
    "premium": [np.nan, np.nan, 295, np.nan, np.nan, 330, 520, np.nan, 540, 515, 505, 530]})
print("Blank rate of premium by plan:")
display(blank_rate(demo, "premium", "plan"))
print("Blank rate of premium by branch:")
display(blank_rate(demo, "premium", "branch"))
print("The same counts as a grid:")
display(pd.crosstab(demo["plan"], demo["premium"].isna().rename("premium blank"), margins=True))

In [ ]:
# Missingness inventory: count and share of blank cells per column of the joined table.
# YOUR CODE HERE

In [ ]:
# The claims table alone: blank rate of the column you found, by each other claims column, with group sizes.
# YOUR CODE HERE

What does the claims table alone tell you about the blanks you found? Name a column only if you can say how much you would trust its difference, given the group sizes.

> Your answer:

In [ ]:
# The joined table: the same blank-rate comparison by each reference-table column.
# YOUR CODE HERE

Which mechanism does your evidence support: MCAR, MAR, MNAR, or can you not tell from the data? Give the evidence with group sizes, and say what the data alone cannot answer.

> Your answer:

### Step A3: Clean

Treat the missingness, duplicates, status codes, impossible values, and type errors you find. Everything happens in code, on copies; `claims` and `policyholders` stay as they arrived. Print a row count after each step that can change the number of rows, so the counts reconcile from start to finish. Build the final table as `cleaned`.

Log each decision in Step A5 as you make it, not afterward. When a rule or a threshold is yours (what counts as a match, which row to keep), write it down in the log.

In [ ]:
# Template: the cleaned table you build in this step. Later cells wait for it.
cleaned = None

In [ ]:
# Cleaning workspace: all treatments in code, on copies, with a row count after each step that can change the count. End with the variable cleaned.
# YOUR CODE HERE

In [ ]:
# Given: the shape of the cleaned table and its first rows.
if ready(cleaned, "the cleaning in Step A3"):
    print(cleaned.shape[0], "rows,", cleaned.shape[1], "columns")
    display(cleaned.head())

In [ ]:
# Optional: write the cleaned table to a file. Colab storage is temporary, so download it if you want to keep it.
# cleaned.to_csv("cleaned_claims.csv", index=False)

### Step A4: The imputation-distortion experiment

Fill the missing `annual_income` values three ways, on copies of the cleaned income column: with the overall mean, with the overall median, and with the median within each customer segment. Leave the blanks in `cleaned` as they are. Then compare the three filled versions with the observed distribution.

The plotting helper and the summary table are given. The three fills are yours.

In [ ]:
# Given: plot_income_overlay draws the observed and filled distributions on one set of axes and returns the figure.
def plot_income_overlay(series_by_label):
    """Overlay step histograms (density scale, shared bins). Each series has its own color and line style."""
    styles = ["-", "--", ":", "-."]
    edges = np.arange(0, 440_000, 10_000)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for (label, values), style in zip(series_by_label.items(), styles):
        ax.hist(values.dropna(), bins=edges, density=True, histtype="step", linewidth=2, linestyle=style, label=label)
    ax.set_title("Annual income: observed values and three fills")
    ax.set_xlabel("annual income (dollars)")
    ax.set_ylabel("density")
    ax.legend()
    return fig

In [ ]:
# Template: the three filled income columns you build in this step. Later cells wait for them.
income_mean = income_median = income_group = None

In [ ]:
# The three fills: mean, median, and group median by customer segment, each on a copy of the cleaned income column.
# YOUR CODE HERE

In [ ]:
# Given: a summary table of the four versions, then the overlay figure.
needed = {"the cleaned table (Step A3)": cleaned, "the mean fill": income_mean,
          "the median fill": income_median, "the group-median fill": income_group}
missing = [name for name, value in needed.items() if value is None]
if missing:
    print("Not built yet: " + "; ".join(missing) + ". This cell runs once they exist.")
else:
    # Each fill may be a pandas Series, a numpy array (np.where returns one) or a list; all become Series here.
    versions = {label: pd.Series(values, dtype="float64") for label, values in
                {"observed": cleaned["annual_income"], "mean fill": income_mean,
                 "median fill": income_median, "group-median fill": income_group}.items()}
    summary = pd.DataFrame({label: {"n": values.notna().sum(), "mean": values.mean(), "median": values.median(),
                                    "std": values.std(), "skewness": values.skew()}
                            for label, values in versions.items()}).T.round(2)
    display(summary)
    fig = plot_income_overlay(versions)
    plt.show()

Figure: overlay of four income distributions, observed values and the mean, median and group-median fills, each with its own line style and color.

Write four sentences: what the mean fill did to the shape, what the median fill did, what the group-median fill did, and who downstream would be misled by which choice.

> Your answer:

### Step A5: The decision log

One row per issue. The six columns are the structure of your log and of the rubric that grades it:

| Column | What goes in it |
|---|---|
| issue | what is wrong, in one line |
| columns affected | the columns the issue touches |
| mechanism and evidence | where applicable, the mechanism you hypothesize and the evidence in the data for it; say where the data alone cannot answer |
| treatment | exactly what you did, with any rule or threshold you chose |
| alternatives rejected | what else you could have done, and why you did not |
| information lost | what the treatment destroyed; "None" only when it is true |

A defensible decision that destroys information beats an indefensible one that pretends not to.

Add a row with `log_decision(...)`; call `show_log()` at the end of your cell to see the log so far. Here is an example on an invented issue:

```python
log_decision(
    issue="Branch code typed two ways in an invented table",
    columns="branch",
    mechanism_and_evidence="Not a missingness issue. value_counts() shows N-07 and N07 as separate labels.",
    treatment="Replaced N07 with N-07 on 3 rows",
    alternatives_rejected="Drop the 3 rows (rejected: the rest of each row is valid)",
    information_lost="None: the label was a spelling variant")
```

In [ ]:
# Decision log template: log_decision(...) adds one row; show_log() displays the log so far.
LOG_COLUMNS = ["issue", "columns affected", "mechanism and evidence", "treatment", "alternatives rejected", "information lost"]
decision_log = []

def log_decision(issue, columns, mechanism_and_evidence, treatment, alternatives_rejected, information_lost):
    """Add one row to the decision log."""
    values = [issue, columns, mechanism_and_evidence, treatment, alternatives_rejected, information_lost]
    decision_log.append(dict(zip(LOG_COLUMNS, values)))

def show_log():
    """Display the decision log as a table."""
    if not decision_log:
        print("The log has no rows yet. Add one with log_decision(...).")
        return
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(decision_log))

In [ ]:
# Your decision log rows: one log_decision(...) call per issue.
decision_log.clear()   # empties the log first, so running this cell again does not add every row twice
# YOUR CODE HERE

In [ ]:
# Given: check the decision log. It lists blank fields and never stops the notebook.
if not decision_log:
    print("No rows to check yet.")
else:
    def _blank(value):
        """True for a missing entry: None, NaN, or text that is only spaces."""
        return value is None or (isinstance(value, float) and value != value) or not str(value).strip()
    print(len(decision_log), "row(s) in the log.")
    blanks = [(i + 1, col) for i, row in enumerate(decision_log) for col in LOG_COLUMNS if _blank(row[col])]
    if blanks:
        for number, col in blanks:
            print(f"Row {number} has no entry in: {col}")
    else:
        print("Every field of every logged row is filled in.")

### Step A6: Re-profile and three sentences

Profile the raw joined table and the cleaned table with the same tool. Each report takes about half a minute and is saved as an HTML file you can download and open. The comparison cell prints what changed, without being told what to look for.

In [ ]:
# Template: the two profile reports, and a helper that lists a report's alerts as text.
before_report = after_report = None

def alert_names(report):
    """List the alerts of a profile report as 'TYPE: column' strings."""
    return sorted(f"{a.alert_type.name}: {a.column_name}" if a.column_name else a.alert_type.name
                  for a in report.description_set.alerts)

In [ ]:
# Given: profile the joined table as it arrived, save the full report as an HTML file, and list its alerts.
if ready(joined_raw, "the join in Step A1"):
    from data_profiling import ProfileReport
    started = time.time()
    before_frame = joined_raw.drop(columns="_merge", errors="ignore")
    before_report = ProfileReport(before_frame, title="Joined table, as it arrived", progress_bar=False)
    before_report.to_file("m3_profile_before.html")
    print(f"Saved m3_profile_before.html in {time.time() - started:.0f} s. Alerts:")
    print(*alert_names(before_report), sep="\n")

In [ ]:
# Given: profile the cleaned table the same way and list its alerts.
if ready(cleaned, "the cleaning in Step A3"):
    from data_profiling import ProfileReport
    started = time.time()
    after_report = ProfileReport(cleaned, title="Cleaned table", progress_bar=False)
    after_report.to_file("m3_profile_after.html")
    print(f"Saved m3_profile_after.html in {time.time() - started:.0f} s. Alerts:")
    print(*alert_names(after_report), sep="\n")

In [ ]:
# Given: a generic before/after comparison of the two profiles and the two tables.
if ready(before_report, "the profile of the joined table") and ready(after_report, "the profile of the cleaned table"):
    tb, ta = before_report.description_set.table, after_report.description_set.table
    counts = pd.DataFrame({"before": [tb["n"], tb["n_var"], tb["n_cells_missing"], tb.get("n_duplicates", 0)],
                           "after": [ta["n"], ta["n_var"], ta["n_cells_missing"], ta.get("n_duplicates", 0)]},
                          index=["rows", "columns", "blank cells", "duplicate rows"])
    display(counts)
    before_alerts, after_alerts = alert_names(before_report), alert_names(after_report)
    print("alerts only before:", [a for a in before_alerts if a not in after_alerts])
    print("alerts only after: ", [a for a in after_alerts if a not in before_alerts])
    print("numeric columns whose minimum or maximum changed:")
    for col in before_frame.select_dtypes("number").columns.intersection(cleaned.columns):
        if not pd.api.types.is_numeric_dtype(cleaned[col]):
            print(f"  {col}: no longer numeric in cleaned (now {cleaned[col].dtype})")
            continue
        old = (float(before_frame[col].min()), float(before_frame[col].max()))
        new = (float(cleaned[col].min()), float(cleaned[col].max()))
        if old != new:
            print(f"  {col}: min and max {old} -> {new}")

Write three sentences on what the before/after comparison confirms, and one thing it does not show.

> Your answer:

## Part B: AI-assisted

About 1 hour. Part B grades your audit, not the AI's output. The AI's script is raw material and is not graded. Points attach primarily to the critique memo and secondarily to the disposition-of-output table (decisions that hold up against what you learned in Part A, reasons you can check, consistency with this notebook). The other disclosure fields are graded for completion.

Before starting, reread your week 1 AI Working Agreement and notice where this assignment sits relative to your own rules. You are not asked to submit anything about it, but the comparison is the beginning of the revision you will write in week 9.

### Step B1: Load the raw tables again

Part B starts from the raw tables, not from your Part A work. The cell below loads fresh copies into new variables so nothing from Part A is touched.

In [ ]:
# Fresh load of both raw tables for Part B, into new variables so nothing from Part A is touched.
claims_b = load("m3_claims_extract.csv")
policyholders_b = load("m3_policyholders.csv")
print(claims_b.shape, policyholders_b.shape)

### Step B2: Get the AI's proposal and run it

Follow [the AI Protocol page](https://github.com/nybabot/ERM424/blob/main/docs/ai_protocol/ai_protocol.md). Give your assistant [the Module 3 role prompt](https://github.com/nybabot/ERM424/blob/main/docs/ai_protocol/role_prompt_m3.md) exactly as provided, then ask it to propose a cleaning script for the two raw tables. The role prompt does not include the data dictionary, on purpose: what you know about this data is your advantage as the auditor.

Paste the script into the cell below and run it on `claims_b` and `policyholders_b`, or in a fresh copy of this notebook (File, then Save a copy in Drive). Do not overwrite anything from Part A.

In [ ]:
# Paste the AI's proposed cleaning script below this line. The cell does nothing until you do.

### Step B3: The critique memo

This is what is graded. Identify at least two things the AI's approach silently got wrong or assumed without disclosure, judged against what you learned in Part A, and at least one thing it did better or faster than you. Concise, specific, and verifiable beats vague and verbose. For each item, say what you checked and how someone else could check it.

**Silently wrong or undisclosed, item 1.**

> Your answer:

**Silently wrong or undisclosed, item 2.**

> Your answer:

**Done better or faster than you.**

> Your answer:

### Step B4: The disclosure form

Complete every field. It should take about five minutes.

**Tool and access path.** Which AI assistant you used and how you reached it (for example, Colab's built-in assistance, or a named external tool through web chat).

> Your answer:

**Role prompt confirmation.** Confirm that you used the standard role prompt exactly as provided. If you did not, say what differed.

> Your answer:

**Additional prompts.** Your own prompts and follow-up instructions beyond the role prompt, listed or pasted.

> Your answer:

**Additional context shared.** Anything you gave the assistant beyond the prompts themselves (for example, the data dictionary or excerpts of the data).

> Your answer:

**Disposition of output.** One row for each operation in the AI's proposal: the operation, your decision (accepted, modified, or rejected), and a one-line reason you can check. Add rows with `log_disposition(operation, decision, reason)` and call `show_dispositions()` to see the table; for example `log_disposition("convert date text to dates", "accepted", "parses cleanly, checked with min() and max()")`.

In [ ]:
# Disposition table template: log_disposition(...) adds one row; show_dispositions() displays the table.
DECISIONS = ["accepted", "modified", "rejected"]
dispositions = []

def log_disposition(operation, decision, reason):
    """Add one row: an operation in the AI's proposal, your decision on it, and a one-line reason."""
    if decision not in DECISIONS:
        print(f"Not added: decision must be one of {DECISIONS}, not {decision!r}.")
        return
    dispositions.append({"operation": operation, "decision": decision, "reason": reason})

def show_dispositions():
    """Display the disposition table."""
    if not dispositions:
        print("The table has no rows yet. Add one with log_disposition(...).")
        return
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(dispositions))

In [ ]:
# Your disposition rows: one log_disposition(...) call per operation in the AI's proposal.
dispositions.clear()   # empties the table first, so running this cell again does not add every row twice
# YOUR CODE HERE

In [ ]:
# Given: check the disposition table. It lists blank reasons and counts rows by decision, and never stops the notebook.
if not dispositions:
    print("No rows to check yet.")
else:
    def _blank(value):
        """True for a missing entry: None, NaN, or text that is only spaces."""
        return value is None or (isinstance(value, float) and value != value) or not str(value).strip()
    blank_reasons = [i + 1 for i, row in enumerate(dispositions) if _blank(row["reason"])]
    print("rows with no reason:", blank_reasons if blank_reasons else "none")
    display(pd.Series([row["decision"] for row in dispositions]).value_counts().rename("rows").to_frame())

**Attestation.** One sentence confirming that the disclosure above is complete and accurate.

> Your answer:

## Deliverables checklist

Before you submit, confirm that this notebook, saved in your copy, shows all of the following:

1. Part A: the join with before and after evidence, the missingness investigation with your mechanism and evidence, the cleaning in code, the imputation-distortion experiment with your four sentences, the decision log with every field filled in, and the re-profile with your three sentences.
2. Part B: your critique memo, and the disclosure form with all six fields complete, including the disposition table.
3. All cell outputs saved: run the notebook from the top before you share it.

## Submit

1. Save your copy to Google Drive if you have not already: File, then Save a copy in Drive.
2. Set the share link so your instructor can view it: Share, then under General access choose Anyone with the link, Viewer.
3. Paste the link into the Canvas assignment.

If anything fails, post in the Q&A discussion or bring it to office hours.